<a href="https://colab.research.google.com/github/Rafael-Labarrere/Machine-Learning/blob/Trabalho-Final/trabalho_final_igor_rafael_v03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Trabalho Final - Inteligência Artificial
## 📌 Tema: Modelo para Prever o Salário em vagas de T.I e sua evolução ao longo do tempo

#👾 CÓDIGO:

In [1]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [ ]:
# Instalação das dependências atualizadas com suporte a PyTorch (padrão mais estável para HuggingFace)
%pip install transformers torch geopy country-converter scikit-learn --quiet

import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel

# Definição das constantes globais para evitar erros de 'not defined' nas células subsequentes
MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 128


Note: you may need to restart the kernel to use updated packages.


C:\Users\Rafael\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Agora faça os imports
from geopy.geocoders import Nominatim
import country_converter as coco
from sklearn.utils import resample
from tensorflow.keras.layers import GlobalAveragePooling1D
from tensorflow.keras.regularizers import l1_l2

In [ ]:
# Estas variáveis DEVEM estar definidas antes:
MODEL_NAME = 'bert-base-uncased'  # ou outro modelo BERT
MAX_LEN = 128                     # ou outro valor adequado
# X_structured_train_processed deve existir do pré-processamento

# Passo 1: Ajustes no Pré-processamento


In [ ]:
# ==============================================================================
# 1. IMPORTS E CONFIGURAÇÕES
# ==============================================================================
import pandas as pd
import numpy as np
import torch
from transformers import BertTokenizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.utils import resample
import country_converter as coco
from geopy.geocoders import Nominatim
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações visuais
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 7)

# ==============================================================================
# 2. CARREGAMENTO DOS DADOS (COM TRATAMENTO DE CAMINHO)
# ==============================================================================
# Tenta carregar de ambos os caminhos para funcionar para Ígor e Rafael
try:
    # ENDEREÇO *ÍGOR*:
    df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/IA/trabalho-final/ai_job_dataset.csv')
    print("Dataset carregado com sucesso do caminho do Ígor.")
except FileNotFoundError:
    try:
        # ENDEREÇO *RAFAEL*:
        df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/ai_job_dataset.csv')
        print("Dataset carregado com sucesso do caminho do Rafael.")
    except FileNotFoundError:
        print("Erro: O arquivo 'ai_job_dataset.csv' não foi encontrado em nenhum dos caminhos. Verifique o Google Drive.")
        # Interrompe a execução se o arquivo não for encontrado
        exit()

# ==============================================================================
# NOVO CÓDIGO - ENRIQUECIMENTO DE FEATURES GEOGRÁFICAS
# ==============================================================================
def enrich_location_data(df):
    # Padronizar códigos de país
    df['country_code'] = coco.convert(names=df['company_location'], to='ISO3')
    df['continent'] = coco.convert(names=df['company_location'], to='continent')

    # Exemplo simplificado - na prática use dados reais
    gdp_map = {'USA': 1.0, 'GBR': 0.8, 'IND': 0.2}
    df['gdp_weight'] = df['country_code'].map(gdp_map).fillna(0.5)

    return df

df = enrich_location_data(df)

# ==============================================================================
# 3. PRÉ-PROCESSAMENTO E ENGENHARIA DE FEATURES
# ==============================================================================

# 3.1. Combinação de campos textuais e seleção de features
df['text_input'] = df['job_title'] + " " + df['required_skills']

categorical_features = ['experience_level', 'employment_type', 'company_location',
                        'company_size', 'employee_residence', 'education_required',
                        'industry', 'salary_currency', 'company_name', 'job_title']
numerical_features = ['remote_ratio', 'years_experience', 'job_description_length', 'benefits_score']
text_feature = 'text_input'
target = 'salary_usd'

# 3.2. Tratamento de Valores Ausentes
# Para colunas numéricas, preenchemos com a mediana (mais robusta a outliers)
for col in numerical_features:
    median_val = df[col].median()
    df[col].fillna(median_val, inplace=True)

# Para colunas categóricas, preenchemos com a moda (valor mais frequente)
for col in categorical_features:
    mode_val = df[col].mode()[0]
    df[col].fillna(mode_val, inplace=True)

print("\nTratamento de valores ausentes concluído.")
print(f"Valores nulos restantes no dataframe: {df.isnull().sum().sum()}")

# ==============================================================================
# NOVO CÓDIGO - BALANCEAMENTO POR REGIÃO
# ==============================================================================
def balance_by_region(df, region_col='continent'):
    regions = df[region_col].unique()
    min_size = df[region_col].value_counts().min()

    balanced_dfs = []
    for region in regions:
        region_df = df[df[region_col] == region]
        balanced_dfs.append(
            resample(region_df,
                   replace=len(region_df) < min_size,
                   n_samples=min_size,
                   random_state=42)
        )
    return pd.concat(balanced_dfs)

df_balanced = balance_by_region(df)

# 3.3. Divisão dos dados em Treino e Validação
X = df_balanced.drop(target, axis=1)
y = df_balanced[target]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 3.4. Processamento das Features Estruturadas (Numéricas e Categóricas)
# Usaremos um ColumnTransformer que aplica os transformadores corretos a cada tipo de coluna.
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor_structured = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'
)

# Ajustamos o preprocessador SOMENTE nos dados de treino
X_structured_train_processed = preprocessor_structured.fit_transform(X_train)
# E aplicamos a transformação nos dados de validação
X_structured_val_processed = preprocessor_structured.transform(X_val)

# 3.5. Tokenização dos Textos com BertTokenizer
MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 128
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

def bert_tokenizer_func(texts, tokenizer, max_len):
    return tokenizer(
        texts.tolist(),
        add_special_tokens=True,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt' # Retorna tensores do PyTorch (Versão corrigida)
    )

# Tokenizando os textos de treino e validação
train_encodings = bert_tokenizer_func(X_train[text_feature], tokenizer, MAX_LEN)
val_encodings = bert_tokenizer_func(X_val[text_feature], tokenizer, MAX_LEN)

print("\nPré-processamento e tokenização concluídos com PyTorch!")
print(f"Shape dos dados estruturados de treino: {X_structured_train_processed.shape}")
print(f"Shape dos dados de texto (input_ids) de treino: {train_encodings['input_ids'].shape}")


Dataset carregado com sucesso do caminho do Rafael.


<ipython-input-4-235683164>:74: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(median_val, inplace=True)
<ipython-input-4-235683164>:79: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({


Tratamento de valores ausentes concluído.
Valores nulos restantes no dataframe: 0


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]


Pré-processamento e tokenização concluídos!
Shape dos dados estruturados de treino: (2342, 113)
Shape dos dados de texto (input_ids) de treino: (2342, 128)


# Passo 2: Correção da Arquitetura do Modelo

In [ ]:
# ==============================================================================
# 4. CRIAÇÃO DO MODELO HÍBRIDO
# ==============================================================================
from transformers import TFBertModel
from tensorflow.keras.layers import Input, Dense, Concatenate, Dropout, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l1_l2

def create_hybrid_bert_model(bert_model_name, max_len, num_structured_features):
    # Camadas de entrada
    input_ids = Input(shape=(max_len,), dtype=tf.int32, name='input_ids')
    attention_mask = Input(shape=(max_len,), dtype=tf.int32, name='attention_mask')
    structured_input = Input(shape=(num_structured_features,), name='structured_input')

    # Carregar BERT
    bert_model = TFBertModel.from_pretrained(bert_model_name)

    # Obter outputs do BERT
    bert_output = bert_model(
        input_ids=input_ids,
        attention_mask=attention_mask
    ).last_hidden_state

    # Pooling e combinação
    text_features = GlobalAveragePooling1D()(bert_output)
    combined = Concatenate()([text_features, structured_input])

    # Camadas densas
    x = Dense(128, activation='relu', kernel_regularizer=l1_l2(0.01, 0.01))(combined)
    x = Dropout(0.3)(x)
    output = Dense(1, activation='linear')(x)

    return Model(
        inputs=[input_ids, attention_mask, structured_input],
        outputs=output
    )

# Criar o modelo
hybrid_model = create_hybrid_bert_model(
    MODEL_NAME,
    MAX_LEN,
    X_structured_train_processed.shape[1]
)

# Compilar o modelo
hybrid_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-5),
    loss='mean_squared_error',
    metrics=['mae', tf.keras.metrics.RootMeanSquaredError()]
)

# Mostrar resumo
hybrid_model.summary()

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

ValueError: Exception encountered when calling layer 'tf_bert_model_10' (type TFBertModel).

Data of type <class 'keras.src.backend.common.keras_tensor.KerasTensor'> is not allowed only (<class 'tensorflow.python.framework.tensor.Tensor'>, <class 'bool'>, <class 'int'>, <class 'transformers.utils.generic.ModelOutput'>, <class 'tuple'>, <class 'list'>, <class 'dict'>, <class 'numpy.ndarray'>) is accepted for attention_mask.

Call arguments received by layer 'tf_bert_model_10' (type TFBertModel):
  • input_ids=<KerasTensor shape=(None, 128), dtype=int32, sparse=False, name=input_ids>
  • attention_mask=<KerasTensor shape=(None, 128), dtype=int32, sparse=False, name=attention_mask>
  • token_type_ids=None
  • position_ids=None
  • head_mask=None
  • inputs_embeds=None
  • encoder_hidden_states=None
  • encoder_attention_mask=None
  • past_key_values=None
  • use_cache=None
  • output_attentions=None
  • output_hidden_states=None
  • return_dict=None
  • training=False

In [ ]:
# ==============================================================================
# 4. CRIAÇÃO E RESUMO DO MODELO HÍBRIDO (VERSÃO CORRIGIDA)
# ==============================================================================

# 1. Verificar shape dos dados estruturados
print(f"Shape dos dados estruturados: {X_structured_train_processed.shape}")

# 2. Definir a arquitetura do modelo
def create_hybrid_bert_model(bert_model_name, max_len, num_structured_features):
    # Camadas de entrada
    input_ids = Input(shape=(max_len,), dtype=tf.int32, name='input_ids')
    attention_mask = Input(shape=(max_len,), dtype=tf.int32, name='attention_mask')
    structured_input = Input(shape=(num_structured_features,), name='structured_input')

    # Camada BERT
    bert_model = TFBertModel.from_pretrained(bert_model_name)
    bert_output = bert_model(input_ids, attention_mask=attention_mask).last_hidden_state

    # Pooling e concatenação
    text_features = GlobalAveragePooling1D()(bert_output)
    combined = Concatenate()([text_features, structured_input])

    # Camadas densas
    x = Dense(128, activation='relu', kernel_regularizer=l1_l2(0.01, 0.01))(combined)
    x = Dropout(0.3)(x)
    output = Dense(1, activation='linear')(x)

    return Model(inputs=[input_ids, attention_mask, structured_input], outputs=output)

# 3. Criar e compilar o modelo
try:
    hybrid_model = create_hybrid_bert_model(MODEL_NAME, MAX_LEN, X_structured_train_processed.shape[1])

    hybrid_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=3e-5),
        loss='mse',
        metrics=['mae', tf.keras.metrics.RootMeanSquaredError(name='rmse')]
    )

    # 4. Mostrar resumo detalhado
    print("\n" + "="*80)
    print("RESUMO DO MODELO HÍBRIDO")
    print("="*80)
    hybrid_model.summary(show_trainable=True, line_length=100)

    # Mostrar informações adicionais
    print("\nInformações Adicionais:")
    print(f"- Total de parâmetros: {hybrid_model.count_params():,}")
    print(f"- Parâmetros treináveis: {sum([tf.keras.backend.count_params(w) for w in hybrid_model.trainable_weights]):,}")
    print(f"- Camadas: {len(hybrid_model.layers)}")

except Exception as e:
    print(f"\nERRO ao criar o modelo: {str(e)}")
    print("Verifique se:")
    print("1. Todas as variáveis necessárias estão definidas")
    print("2. Os imports estão corretos")
    print("3. O pré-processamento foi executado completamente")

Shape dos dados estruturados: (2342, 113)


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w


ERRO ao criar o modelo: Exception encountered when calling layer 'tf_bert_model_4' (type TFBertModel).

Data of type <class 'keras.src.backend.common.keras_tensor.KerasTensor'> is not allowed only (<class 'tensorflow.python.framework.tensor.Tensor'>, <class 'bool'>, <class 'int'>, <class 'transformers.utils.generic.ModelOutput'>, <class 'tuple'>, <class 'list'>, <class 'dict'>, <class 'numpy.ndarray'>) is accepted for attention_mask.

Call arguments received by layer 'tf_bert_model_4' (type TFBertModel):
  • input_ids=<KerasTensor shape=(None, 128), dtype=int32, sparse=False, name=input_ids>
  • attention_mask=<KerasTensor shape=(None, 128), dtype=int32, sparse=False, name=attention_mask>
  • token_type_ids=None
  • position_ids=None
  • head_mask=None
  • inputs_embeds=None
  • encoder_hidden_states=None
  • encoder_attention_mask=None
  • past_key_values=None
  • use_cache=None
  • output_attentions=None
  • output_hidden_states=None
  • return_dict=None
  • training=False
Verifique

#

# Passo 3: Treinamento do Modelo

In [ ]:
# ==============================================================================
# 4. CRIAÇÃO DO MODELO HÍBRIDO (VERSÃO CORRIGIDA)
# ==============================================================================

from transformers import TFBertModel
from tensorflow.keras.layers import Input, Dense, Concatenate, Dropout, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l1_l2

def create_hybrid_bert_model(bert_model_name, max_len, num_structured_features):
    # 1. Camadas de entrada
    input_ids = Input(shape=(max_len,), dtype=tf.int32, name='input_ids')
    attention_mask = Input(shape=(max_len,), dtype=tf.int32, name='attention_mask')
    structured_input = Input(shape=(num_structured_features,), name='structured_input')

    # 2. Carregar o BERT (com tratamento especial para KerasTensor)
    bert_model = TFBertModel.from_pretrained(bert_model_name)

    # Converter KerasTensor para Tensor regular explicitamente
    input_ids_tensor = tf.cast(input_ids, tf.int32)
    attention_mask_tensor = tf.cast(attention_mask, tf.int32)

    # 3. Obter outputs do BERT
    bert_output = bert_model(
        input_ids=input_ids_tensor,
        attention_mask=attention_mask_tensor
    ).last_hidden_state

    # 4. Pooling e combinação
    text_features = GlobalAveragePooling1D()(bert_output)
    combined = Concatenate()([text_features, structured_input])

    # 5. Camadas densas
    x = Dense(128, activation='relu', kernel_regularizer=l1_l2(0.01, 0.01))(combined)
    x = Dropout(0.3)(x)
    output = Dense(1, activation='linear')(x)

    return Model(
        inputs=[input_ids, attention_mask, structured_input],
        outputs=output,
        name='hybrid_bert_model'
    )

# Criar o modelo
try:
    hybrid_model = create_hybrid_bert_model(
        MODEL_NAME,
        MAX_LEN,
        X_structured_train_processed.shape[1]
    )

    # Compilar o modelo
    hybrid_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=3e-5),
        loss='mean_squared_error',
        metrics=['mae', tf.keras.metrics.RootMeanSquaredError()]
    )

    # Mostrar resumo
    hybrid_model.summary()

except Exception as e:
    print(f"Erro ao criar o modelo: {str(e)}")
    print("Verifique se:")
    print("1. MODEL_NAME está definido corretamente")
    print("2. MAX_LEN é um inteiro válido")
    print("3. X_structured_train_processed existe e tem o shape correto")

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

Erro ao criar o modelo: A KerasTensor cannot be used as input to a TensorFlow function. A KerasTensor is a symbolic placeholder for a shape and dtype, used when constructing Keras Functional models or Keras Functions. You can only use it as input to a Keras layer or a Keras operation (from the namespaces `keras.layers` and `keras.operations`). You are likely doing something like:

```
x = Input(...)
...
tf_fn(x)  # Invalid.
```

What you should do instead is wrap `tf_fn` in a layer:

```
class MyLayer(Layer):
    def call(self, x):
        return tf_fn(x)

x = MyLayer()(x)
```

Verifique se:
1. MODEL_NAME está definido corretamente
2. MAX_LEN é um inteiro válido
3. X_structured_train_processed existe e tem o shape correto


In [ ]:
# ==============================================================================
# 5. TREINAMENTO DE VERIFICAÇÃO (50 PASSOS)
# ==============================================================================

# (O código para preparar X_train_inputs e X_val_inputs continua o mesmo)
X_train_inputs = {
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'structured_input': X_structured_train_processed
}
X_val_inputs = {
    'input_ids': val_encodings['input_ids'],
    'attention_mask': val_encodings['attention_mask'],
    'structured_input': X_structured_val_processed
}

# Parâmetros para o teste rápido
EPOCHS = 1      # Vamos rodar apenas uma fração da primeira época
BATCH_SIZE = 16

print("\nIniciando um TREINAMENTO DE VERIFICAÇÃO (50 passos)...")

# Treina por apenas 50 passos para um teste rápido
history_teste = hybrid_model.fit(
    X_train_inputs,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_inputs, y_val),
    steps_per_epoch=50, # <<< RODA APENAS 50 PASSOS
    verbose=1
)

print("\nVerificação concluída com sucesso!")


Iniciando um TREINAMENTO DE VERIFICAÇÃO (50 passos)...


NameError: name 'hybrid_model' is not defined

## 2.2 MODELO COMPLETO E MAIS SEGURO:

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

# ==============================================================================
# 5. TREINAMENTO COMPLETO E SEGURO
# ==============================================================================

# (O código para preparar X_train_inputs e X_val_inputs continua o mesmo)
X_train_inputs = {
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'structured_input': X_structured_train_processed
}
X_val_inputs = {
    'input_ids': val_encodings['input_ids'],
    'attention_mask': val_encodings['attention_mask'],
    'structured_input': X_structured_val_processed
}

# Parâmetros do treinamento completo
EPOCHS = 5
BATCH_SIZE = 16

# Callback para salvar o melhor modelo durante o treino
checkpoint_filepath = 'melhor_modelo_pesos.weights.h5'
model_checkpoint_callback = ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True,
    monitor='val_loss', # Salva com base na perda de validação
    mode='min',
    save_best_only=True # Salva apenas o melhor resultado
)

print("\nIniciando o treinamento COMPLETO do modelo...")

# Treinamento final
history = hybrid_model.fit(
    X_train_inputs,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_inputs, y_val),
    verbose=1,
    callbacks=[model_checkpoint_callback] # Adiciona o callback para salvar
)

print("\nTreinamento completo concluído!")
print(f"Os pesos do melhor modelo foram salvos em '{checkpoint_filepath}'.")

# Opcional: carregar os melhores pesos ao final do treinamento
hybrid_model.load_weights(checkpoint_filepath)

# PASSO 4: Avaliação e Visualização dos Resultados

In [ ]:
# ==============================================================================
# 6. AVALIAÇÃO E VISUALIZAÇÃO DOS RESULTADOS (VERSÃO PARA TESTE)
# ==============================================================================

# ==============================================================================
# NOVO CÓDIGO - AVALIAÇÃO POR REGIÃO
# ==============================================================================
def evaluate_by_region(model, X_val, y_val, df_val, region_col='continent'):
    # Preparar DataFrame de validação
    df_val = df_val.copy()
    df_val['salary_usd'] = y_val.values

    results = []
    for region in df_val[region_col].unique():
        mask = df_val[region_col] == region
        X_region = {
            'input_ids': X_val['input_ids'][mask],
            'attention_mask': X_val['attention_mask'][mask],
            'structured_input': X_val['structured_input'][mask]
        }
        y_region = y_val[mask]

        loss, mae, rmse = model.evaluate(X_region, y_region, verbose=0)
        results.append({
            'region': region,
            'samples': mask.sum(),
            'mae': mae,
            'rmse': rmse,
            'avg_salary': y_region.mean()
        })

    return pd.DataFrame(results)

# Obter o DataFrame original de validação
df_val = X_val.copy()
region_results = evaluate_by_region(hybrid_model, X_val_inputs, y_val, df_val)

print("\nResultados por Região:")
print(region_results.sort_values('mae'))

# Plot adicional
plt.figure(figsize=(10, 6))
sns.barplot(data=region_results, x='region', y='mae')
plt.title('MAE por Região')
plt.ylabel('Erro Médio Absoluto (USD)')
plt.xticks(rotation=45)
plt.show()